# Ciber-Inmunología Swarm en Google Colab

Este notebook corre el proyecto completo en Colab usando su GPU. Los nodos son simulados, así que toda la federación ocurre acá; al final quedan en tu Google Drive los resultados (CSV y PNG) y los pesos de los modelos (`.pt`).

**Antes de empezar**

1. Activá la GPU: menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**.
2. En tu Google Drive creá la carpeta `ciber-inmunologia/raw` y subí ahí los 8 CSV de CIC-IDS2017 (*MachineLearningCVE*). Si la carpeta está vacía, el proyecto usa datos sintéticos de prueba.

Ejecutá las celdas en orden.

## 1. Comprobar la GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Descargar el código e instalar Flower

In [ ]:
import os

%cd /content
if not os.path.exists("ciber-inmunologia"):
    !git clone -q https://github.com/xeviercr4/ciber-inmunologia.git
%cd /content/ciber-inmunologia
!git pull -q
!pip install -q -r requirements-colab.txt

## 3. Conectar Google Drive

Colab va a pedir permiso para acceder a tu Drive. Los resultados se guardan directamente ahí, así no se pierden si la sesión se corta.

In [ ]:
import glob
import shutil
from google.colab import drive

drive.mount("/content/drive")

CARPETA_DRIVE = "/content/drive/MyDrive/ciber-inmunologia"
os.makedirs(f"{CARPETA_DRIVE}/raw", exist_ok=True)
os.makedirs(f"{CARPETA_DRIVE}/resultados", exist_ok=True)

# Los experimentos leen esta variable y guardan sus resultados en Drive
os.environ["CIBER_RESULTADOS"] = f"{CARPETA_DRIVE}/resultados"

# Copia los CSV al disco de Colab (leerlos desde ahí es mucho más rápido que desde Drive)
os.makedirs("data/raw", exist_ok=True)
archivos = sorted(glob.glob(f"{CARPETA_DRIVE}/raw/*.csv"))
for archivo in archivos:
    shutil.copy(archivo, "data/raw/")
print(f"CSV copiados: {len(archivos)}")
if not archivos:
    print("No hay CSV en Drive: se van a usar datos SINTÉTICOS de prueba.")

## 4. Verificar el entorno

Tiene que decir `Dispositivo: cuda`.

In [ ]:
!python verificar_entorno.py

## 5. Preprocesar los datos

`MUESTRA` es la fracción del dataset que se usa. Empezá con `0.1` (10%) para medir tiempos y subila después.

In [ ]:
MUESTRA = 0.1
!python -m src.preprocess --sample $MUESTRA

## 6. Experimentos

Elegí el modelo (`mlp` es rápido; `transformer` es el modelo principal y tarda más) y la cantidad de rondas. Cada experimento guarda su CSV y su gráfica en Drive al terminar.

Ejecutá un experimento por vez: no se pueden correr dos simulaciones al mismo tiempo.

In [ ]:
MODELO = "transformer"
RONDAS = 20

In [ ]:
# E1 — centralizado vs. aislado vs. federado
!python -m experiments.e1_comparacion --modelo $MODELO --rondas $RONDAS

In [ ]:
# E2 — FedAvg vs. FedProx vs. cabezas locales, con alpha = 0.1, 0.5 y 1.0
!python -m experiments.e2_heterogeneidad --modelo $MODELO --rondas $RONDAS

In [ ]:
# E3 — zero-day simulado (se puede cambiar la familia: DoS, DDoS, PortScan, BruteForce, Web, Bot)
!python -m experiments.e3_zero_day --modelo $MODELO --rondas $RONDAS --familia DDoS

In [ ]:
# E4 — 1 a 3 nodos maliciosos: FedAvg vs. Krum vs. mediana
!python -m experiments.e4_maliciosos --modelo $MODELO --rondas $RONDAS

## 7. Ver las gráficas

In [ ]:
from IPython.display import Image, display

for ruta in sorted(glob.glob(f"{CARPETA_DRIVE}/resultados/*.png")):
    print(os.path.basename(ruta))
    display(Image(ruta, width=900))

## 8. Llevar los resultados a tu computadora

Todo quedó en tu Drive, en `ciber-inmunologia/resultados` (los pesos de los modelos están en la subcarpeta `modelos`). Esta celda además arma un zip y lo descarga.

Para evaluar en tu máquina un modelo entrenado acá, copiá el `.pt` y ejecutá:

```
python -m src.centralized --evaluar ruta/al/archivo.pt
```

Importante: tu máquina tiene que haber preprocesado los mismos datos con la misma muestra (`--sample`) que se usó acá, porque el modelo depende de las columnas y de la normalización.

In [ ]:
from google.colab import files

shutil.make_archive("/content/resultados", "zip", f"{CARPETA_DRIVE}/resultados")
files.download("/content/resultados.zip")